<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">

# **Operaciones de Aprendizaje Automático III**
# **Clase 3: Ejercicio, MLflow**


 * Entrenar un modelo no es LLMOps
 * LLMOps es lo que hace que ese modelo se pueda volver a encontrar, comparar,
promover y revertir dentro de seis meses, cuando quien lo entrenó ya no está.


## 1. Persistencia y portabilidad del registro

MLflow necesita dos cosas:
* **Backend store:** Experimentos, runs, parametros, métricas, etiquetas. Se guarda en "mlflow.db" (sqlite)
* **artifact store:** archivos: modelos, graficos, json, etc. Se guarda en "mlruns/"

Usamos sqlite porque el registro de modelos necesita una base de datos.

La base y los artefactos son dos lugares distintos. Si solo llevamos solo `mlflow.db`, tenemos los números pero los archivos quedan apuntando a rutas que ya no existen

In [1]:
!pip -q install -U "mlflow==2.21.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.2/28.2 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.8/222.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/

In [2]:
import mlflow, json, os, math, re, random, hashlib
from mlflow.tracking import MlflowClient

Utilizamos una base de datos SQLite llamada mlflow.db como backend para guardar la información de tracking

In [3]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")  # backend store

Creamos un experimento llamado: clase3-previo-mlflow

In [4]:
mlflow.set_experiment("clase3-previo-mlflow")  # agrupa runs relacionados

2026/09/15 22:42:49 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/15 22:42:49 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step
INFO  [alembic.runtime.migration] Running upgrade 451aebb31d03 -> 90e64c465722, migrate user column to tags
INFO  [alembic.runtime.migration] Running upgrade 90e64c465722 -> 181f10493468, allow nulls for metric values
INFO  [alembic.runtime.migration] Running upgrade 181f10493468 -> df50e92ffc5e, Add Experiment Tags Table
INFO  [alembic.runtime.migration] Running upgrade df50e92ffc5e -> 7ac759974ad8, Update run tags with larger limit
INFO  [alembic.runtime.migration] Running upgrade 7ac759974ad8 -> 89d4b8295536, create latest metrics table
INFO  [89d4b8295536_create_latest_metrics_table_py] Migration complete!
INFO  

<Experiment: artifact_location='/content/mlruns/1', creation_time=1789512170796, experiment_id='1', last_update_time=1789512170796, lifecycle_stage='active', name='clase3-previo-mlflow', tags={}>

Creamos un objeto que permite consultar y administrar MLflow

In [5]:
cliente = MlflowClient()
print("todo registrado va a parar a:", mlflow.get_tracking_uri())

todo registrado va a parar a: sqlite:///mlflow.db


## 2. Reproducibilidad (entradas) y comparabilidad (salidas)

Un run es una ejecución, dentro de un run se registran dos cosas:

- Parámetro (log_param): una entrada del experimento, es lo que elegimos antes de correr: la tasa de aprendizaje, la semilla, el modelo base, la plantilla,no cambia durante el run, y sirve para reproducir. **Lo que escribimos antes de ejecutar**.

- Métrica (log_metric): una salida, es un número que sale de correr: la pérdida, el acierto, los segundos, puede tener varios valores a lo largo del tiempo, y sirve para comparar. **Lo que produjo la corrida**

In [6]:
#creamos una nueva ejecución llamada mi-primer-run
with mlflow.start_run(run_name="mi-primer-run") as run:
    # Los parameters normalmente representan configuraciones que elegimos antes o durante el experimento
    mlflow.log_param("semilla", 42)
    mlflow.log_param("descripcion", "run de prueba, no mide nada real")

    # Registramos una metrica
    mlflow.log_metric("acierto", 0.73)

    # Cada run de MLflow tiene un identificador único
    primer_id = run.info.run_id

print("run_id:", primer_id)

run_id: 26c673d731374d28913f20758fb9a4af


Vamos a ver lo que quedó registrado

In [7]:
r = mlflow.get_run(primer_id)
print("params :", r.data.params)
print("metrics:", r.data.metrics)
print("estado :", r.info.status)

params : {'semilla': '42', 'descripcion': 'run de prueba, no mide nada real'}
metrics: {'acierto': 0.73}
estado : FINISHED


## 3. Tarea de juguete

Instrucciones cuya respuesta se puede calcular en Python, así que la evaluación es exacta.

Tenemos tres "modelos" que en realidad son funciones cortas:
* **verboso**: una oración completa "el resultado es 51"
* **seco**: solo el valor "51"
* **eco**: repite el enunciado y no contesta nada

Los dos primeros resuelven bien la tarea y solo cambian el formato, el tercero no resuelve nada, y es para que veamos cómo una métrica mal elegida igual da puntos.

In [8]:
def suma(a, b):
  return f"Suma {a} y {b}. Responde solo el resultado.", str(a + b)

def mayusculas(f):
  return f"Pon este texto en mayúsculas: {f}", f.upper()

def primera(f):
  return f"Responde solo la primera palabra de: {f}", f.split()[0]

def contar(f):
  return f"Cuántas palabras tiene esta frase? Responde solo el número: {f}", str(len(f.split()))

conjunto de frases que se utilizarán para generar casos de prueba

In [9]:
FRASES = ["el vuelo sale temprano",
          "el informe está listo",
          "no funciona el ascensor",
          "llegaron los repuestos",
          "el tren viene demorado",
          "falta firmar el contrato"]

Construimos un dataset de evaluación pequeño

In [10]:
rng = random.Random(42)
CASOS = ([suma(rng.randint(2, 90), rng.randint(2, 90)) for _ in range(6)] +
         [mayusculas(f) for f in FRASES] +
         [primera(f) for f in FRASES] +
         [contar(f) for f in FRASES])
print(f"{len(CASOS)} casos de evaluación")

24 casos de evaluación


In [11]:
CASOS

[('Suma 83 y 16. Responde solo el resultado.', '99'),
 ('Suma 5 y 37. Responde solo el resultado.', '42'),
 ('Suma 33 y 30. Responde solo el resultado.', '63'),
 ('Suma 19 y 15. Responde solo el resultado.', '34'),
 ('Suma 88 y 71. Responde solo el resultado.', '159'),
 ('Suma 13 y 77. Responde solo el resultado.', '90'),
 ('Pon este texto en mayúsculas: el vuelo sale temprano',
  'EL VUELO SALE TEMPRANO'),
 ('Pon este texto en mayúsculas: el informe está listo',
  'EL INFORME ESTÁ LISTO'),
 ('Pon este texto en mayúsculas: no funciona el ascensor',
  'NO FUNCIONA EL ASCENSOR'),
 ('Pon este texto en mayúsculas: llegaron los repuestos',
  'LLEGARON LOS REPUESTOS'),
 ('Pon este texto en mayúsculas: el tren viene demorado',
  'EL TREN VIENE DEMORADO'),
 ('Pon este texto en mayúsculas: falta firmar el contrato',
  'FALTA FIRMAR EL CONTRATO'),
 ('Responde solo la primera palabra de: el vuelo sale temprano', 'el'),
 ('Responde solo la primera palabra de: el informe está listo', 'el'),
 ('Resp

Los tres modelos de juguete

In [12]:
def m_verboso(instr, correcta):
  return f"El resultado es {correcta}."

def m_seco(instr, correcta):
  return correcta

def m_eco(instr, correcta):
  return instr        # repite y no contesta

In [13]:
MODELOS = {"verboso": m_verboso, "seco": m_seco, "eco": m_eco}

Esta función intenta hacer que las comparaciones sean un poco más robustas

In [14]:
def _n(s): return re.sub(r"\s+", " ", s.strip()).rstrip(".")

Esto devuelve dos métricas:
* La respuesta correcta aparece dentro de la respuesta del modelo?
* La respuesta del modelo es exactamente lo que se esperaba?

In [15]:
def niveles(salida, esperada):
    return (int(_n(esperada) in _n(salida)), int(_n(salida) == _n(esperada)))

Comprobamos

In [16]:
assert niveles("51", "51") == (1, 1)
assert niveles("El resultado es 51.", "51") == (1, 0)
assert niveles("no sé", "51") == (0, 0)

## 4. Trazabilidad

Esta es la forma que vamos a repetir: un run por cada cosa que queremos comparar, con los mismos nombres de parámetro y de métrica en todos.

Además registramos un artefacto: las salidas crudas en un json. Las tasas nos dicen cuánto falló, solo las salidas nos dicen por qué.

* Generamos un identificador único para el conjunto de evaluación.
* Esto nos dice con qué dataset de evaluación se obtuvo una métrica.
* Si modificamos CASOS cambiará el id

In [17]:
CONJUNTO_ID = hashlib.sha256("||".join(i + "|" + r_ for i, r_ in CASOS).encode()).hexdigest()[:12]
print("id del conjunto de evaluación:", CONJUNTO_ID)

id del conjunto de evaluación: fded731e3228


Esto crea un diccionario para después recuperar cada ejecución

In [18]:
corridas = {}

In [19]:
# Recorremos los tres modelos
for nombre, modelo in MODELOS.items():
    with mlflow.start_run(run_name=f"modelo-{nombre}") as run:
        # Ejecutamos el modelo sobre todos los casos
        salidas = [modelo(i, r_) for i, r_ in CASOS]

        # Evaluamos cada respuesta
        n = [niveles(s, r_) for s, (_, r_) in zip(salidas, CASOS)]
        contiene = sum(x[0] for x in n) / len(n)
        exacto   = sum(x[1] for x in n) / len(n)

        # Parámetros: lo que define la corrida.
        mlflow.log_params({"modelo": nombre, "conjunto_id": CONJUNTO_ID,
                           "n_casos": len(CASOS), "semilla": 42})
        # Métricas: lo que salió de correr.
        mlflow.log_metrics({"contiene": contiene, "exacto": exacto,
                            "largo_medio": sum(len(s) for s in salidas)/len(salidas)})
        # Artefacto: las salidas literales.
        with open(f"salidas_{nombre}.json", "w") as f:
            json.dump([{"instr": i, "esperada": r_, "salida": s}
                       for (i, r_), s in zip(CASOS, salidas)], f, ensure_ascii=False, indent=1)
        mlflow.log_artifact(f"salidas_{nombre}.json")

        # guardamos el ID de MLflow asociado al modelo
        corridas[nombre] = run.info.run_id
        print(f"{nombre:<9} contiene={contiene:.2f}  exacto={exacto:.2f}")

verboso   contiene=1.00  exacto=0.00
seco      contiene=1.00  exacto=1.00
eco       contiene=0.25  exacto=0.00


esperado: "el"
respuesta: "devuelve el numero de elementos"

## 5. Comparación

Leemos desde la base: search_runs devuelve una tabla con todos los
runs del experimento, sus parámetros y sus métricas.

Buscamos todos los runs del experimento clase3-previo-mlflow y los ordenamos de mayor a menor según la métrica "exacto"

In [20]:
tabla = mlflow.search_runs(experiment_names=["clase3-previo-mlflow"],
                           order_by=["metrics.exacto DESC"])

elegimos las columnas de que nos interesan

In [21]:
columnas = ["tags.mlflow.runName", "params.modelo", "metrics.contiene", "metrics.exacto",
            "metrics.largo_medio"]

Esto imprime únicamente esas columnas

In [22]:
print(tabla[columnas].to_string(index=False))

tags.mlflow.runName params.modelo  metrics.contiene  metrics.exacto  metrics.largo_medio
        modelo-seco          seco              1.00             1.0             7.250000
         modelo-eco           eco              0.25             0.0            58.708333
     modelo-verboso       verboso              1.00             0.0            24.250000
      mi-primer-run          None               NaN             NaN                  NaN


El primer run aparece con NaN: no registró esas métricas


In [23]:
mejor = tabla.iloc[0]
print("\nmejor run:", mejor["tags.mlflow.runName"])
print("artefactos:", [a.path for a in cliente.list_artifacts(mejor["run_id"])])
print("viven en :", mlflow.get_run(mejor["run_id"]).info.artifact_uri)


mejor run: modelo-seco
artefactos: ['salidas_seco.json']
viven en : /content/mlruns/1/22d033efd9474d44880a86884880ca4b/artifacts


Puntos importantes:

* **verboso** y **seco** tienen el mismo **contiene** y un **exacto** opuesto. Los dos resuelven la tarea, uno obedece el formato y el otro no. Medir solo **exacto** nos hará creer que **verboso** no sabe nada.

* **eco** no contesta absolutamente nada y aun así tiene puntos de **contiene** En las instrucciones del tipo "responde solo la primera palabra de: el vuelo sale temprano", la respuesta ya está en el enunciado, así que repetirlo alcanza. Una métrica que premia copiar la consigna no
está midiendo capacidad.

* **largo_medio** los separa a los tres sin mirar aciertos. Una métrica barata y descriptiva a veces diagnostica más rápido que la métrica principal.


En el siguiente el modelo base se porta como **verboso** y como **eco** a la vez, y el fine-tuning lo convierte en **seco**.

## 6. Observabilidad del entrenamiento

* **log_metric** acepta un argumento **step**
* Si registramos el mismo nombre de métrica con distintos pasos, MLflow guarda una **serie** y la interfaz la dibuja como curva.
* Así es como se registra la pérdida de entrenamiento en el ejercicio siguiente, directamente desde el historial del **Trainer**.

In [24]:
# creamos una nueva ejecución llamada
with mlflow.start_run(run_name="curva-de-ejemplo") as run:
    mlflow.log_param("nota", "pérdida simulada, solo para ver la curva")
    perdida = 2.5
    for paso in range(1, 41):
        perdida = perdida * 0.93 + random.Random(paso).uniform(-0.02, 0.02)
        mlflow.log_metric("perdida", round(perdida, 4), step=paso)
    curva_id = run.info.run_id

# Lee la serie de vuelta
serie = cliente.get_metric_history(curva_id, "perdida")
print(f"{len(serie)} puntos registrados")
print("primeros:", [round(p.value, 3) for p in serie[:5]])
print("últimos :", [round(p.value, 3) for p in serie[-5:]])

40 puntos registrados
primeros: [2.31, 2.167, 2.005, 1.854, 1.729]
últimos : [0.166, 0.162, 0.156, 0.134, 0.123]


## 7. Registro de modelos: versionar y promover

Hasta acá realizamos experimentos. El registro de modelos es otra cosa, es donde decimos que versión es la que va a producción.

Pasos:

1. **register_model** toma un artefacto de un run y lo convierte en una **versión** de un modelo con nombre.

2. **set_model_version_tag** le da los datos que hacen falta para saber qué es, aquí le ponemos el identificador del conjunto con el que se evaluó.
3. **set_registered_model_alias** le pone un alias, por ejemplo **campeon**, el código de producción pide el alias, no el número de versión.
4. Al cargar por alias, se verifica que sea lo que creemos.

In [25]:
MODELO = "juguete-instrucciones"

Promovemo el que mejor va

In [26]:
ganador = "seco"
version = mlflow.register_model(f"runs:/{corridas[ganador]}/salidas_{ganador}.json", MODELO)
cliente.set_model_version_tag(MODELO, version.version, "conjunto_id", CONJUNTO_ID)
cliente.set_model_version_tag(MODELO, version.version, "modelo", ganador)
cliente.set_registered_model_alias(MODELO, "campeon", version.version)

Successfully registered model 'juguete-instrucciones'.
Created version '1' of model 'juguete-instrucciones'.


Cargamos por alias y comprobamos

In [27]:
v = cliente.get_model_version_by_alias(MODELO, "campeon")
print(f"{MODELO} v{v.version} | alias 'campeon' | etiquetas {v.tags}")
assert v.tags["conjunto_id"] == CONJUNTO_ID, \
    "el modelo promovido no corresponde a este conjunto de evaluación"

juguete-instrucciones v1 | alias 'campeon' | etiquetas {'conjunto_id': 'fded731e3228', 'modelo': 'seco'}


## 8. Reversibilidad

Promovemos una segunda versión, la encontramos peor, y volvemos

1. El alias apunta a otra versión, y las versiones anteriores siguen existiendo. Un registro que sobrescribe no permite revertir.
2. La vuelta atrás también se verifica, con la misma comprobación que la promoción. Una reversión a ciegas es otro despliegue a ciegas.
3. Queda registrado por qué se revirtió, sin eso alguien podria voler a promover la versión mala.

Segunda versión: promovemos el modelo verboso, que obedece peor

In [28]:
v2 = mlflow.register_model(f"runs:/{corridas['verboso']}/salidas_verboso.json", MODELO)
cliente.set_model_version_tag(MODELO, v2.version, "conjunto_id", CONJUNTO_ID)
cliente.set_model_version_tag(MODELO, v2.version, "modelo", "verboso")
cliente.set_registered_model_alias(MODELO, "campeon", v2.version)

Registered model 'juguete-instrucciones' already exists. Creating a new version of this model...
Created version '2' of model 'juguete-instrucciones'.


In [29]:
actual = cliente.get_model_version_by_alias(MODELO, "campeon")
print(f"promovido: v{actual.version} ({actual.tags['modelo']})")

promovido: v2 (verboso)


El criterio de reversión lo fijamos antes: si el exacto del promovido es peor que el de la versión anterior, se revierte

In [30]:
def exacto_de(v):
    return mlflow.get_run(cliente.get_model_version(MODELO, v.version).run_id).data.metrics["exacto"]

In [31]:
anterior = cliente.get_model_version(MODELO, str(int(actual.version) - 1))
print(f"exacto v{actual.version}={exacto_de(actual):.2f}  "
      f"vs v{anterior.version}={exacto_de(anterior):.2f}")

exacto v2=0.00  vs v1=1.00


In [32]:
if exacto_de(actual) < exacto_de(anterior):
    cliente.set_registered_model_alias(MODELO, "campeon", anterior.version)
    # el motivo queda en el registro
    cliente.set_model_version_tag(MODELO, actual.version, "revertida",
                                  f"exacto {exacto_de(actual):.2f} < {exacto_de(anterior):.2f}")
    print(f"revertido a v{anterior.version}")

revertido a v1


la misma verificación que en la promoción, sin excepciones

In [33]:
v = cliente.get_model_version_by_alias(MODELO, "campeon")
assert v.tags["conjunto_id"] == CONJUNTO_ID
assert exacto_de(v) == max(exacto_de(actual), exacto_de(anterior)), "se revirtió al peor"
print(f"campeon = v{v.version} ({v.tags['modelo']}) | verificado")

campeon = v1 (seco) | verificado


Las versiones anteriores no desaparecieron: eso es lo que hace posible revertir, los alias estan en el modelo registrado, no en cada versión

In [34]:
alias_de = {v: a for a, v in cliente.get_registered_model(MODELO).aliases.items()}
print("\nhistorial del modelo:")
for mv in sorted(cliente.search_model_versions(f"name='{MODELO}'"), key=lambda m: int(m.version)):
    print(f"  v{mv.version}  {mv.tags.get('modelo','?'):<8} "
          f"alias={alias_de.get(mv.version, '-'):<8} {mv.tags.get('revertida','')}")


historial del modelo:
  v1  seco     alias=campeon  
  v2  verboso  alias=-        exacto 0.00 < 1.00


Empaquetamos todo lo generado en un .zip y levantamos la interfaz web de MLflow para explorar los experimentos

In [35]:
!zip -qr artefactos_mlflow.zip mlflow.db mlruns salidas_*.json

In [36]:
get_ipython().system_raw("mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000 &")

import time
time.sleep(5)

from google.colab import output
output.serve_kernel_port_as_window(5000)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>